**Feature engineering and searching for correlations between columns**

In [82]:
import pandas as pd

In [83]:
df = pd.read_pickle('sample_data/part3.pkl').set_index('Unnamed: 0')
df_copy = df.copy()

In [84]:
df_copy.head(3)

,order id,date,status,fulfilment,sales channel,ship-service-level,style,sku,category,size,asin,courier status,qty,currency,amount,ship-city,ship-state,ship-postal-code,ship-country,b2b
Unnamed: 0,,,,,,,,,,,,,,,,,,,,
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,Unknown,0,INR,647.62,MUMBAI,MAHARASHTRA,400081,IN,False
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,INR,406.00,BENGALURU,KARNATAKA,560085,IN,False
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210,IN,True


Additional date columns were created by extracting details from **Date** column. The purpose is to comfortably create said dimensions while constructing Star Schema.

In [85]:
df_copy['date'] = pd.to_datetime(df_copy['date'])

In [86]:
df_copy['year'] = df_copy['date'].dt.year

In [87]:
df_copy['month'] = df_copy['date'].dt.month
df_copy['day'] = df_copy['date'].dt.day
df_copy['week'] = df_copy['date'].dt.isocalendar().week

In [88]:
df_copy.head()

,order id,date,status,fulfilment,sales channel,ship-service-level,style,sku,category,size,...,amount,ship-city,ship-state,ship-postal-code,ship-country,b2b,year,month,day,week
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,647.62,MUMBAI,MAHARASHTRA,400081,IN,False,2022,4,30,17
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,406.00,BENGALURU,KARNATAKA,560085,IN,False,2022,4,30,17
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,329.00,NAVI MUMBAI,MAHARASHTRA,410210,IN,True,2022,4,30,17
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,753.33,PUDUCHERRY,PUDUCHERRY,605008,IN,False,2022,4,30,17
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,574.00,CHENNAI,TAMIL NADU,600073,IN,False,2022,4,30,17


A derived feature unit_price was created to better represent product pricing and support analytical queries in the warehouse layer.

In [89]:
df_copy[['qty', 'amount']]
df_copy['qty'].value_counts()

df_copy['unit_price'] = df_copy['qty'] * df_copy['amount']

In [90]:
df_copy[['qty', 'amount', 'unit_price']]

,qty,amount,unit_price
Unnamed: 0,,,
0,0,647.62,0.0
1,1,406.00,406.0
2,1,329.00,329.0
3,0,753.33,0.0
4,1,574.00,574.0
...,...,...,...
128964,1,517.00,517.0
128965,1,999.00,999.0
128966,1,690.00,690.0


B2B and B2C

In [91]:
df_copy.head()

,order id,date,status,fulfilment,sales channel,ship-service-level,style,sku,category,size,...,ship-city,ship-state,ship-postal-code,ship-country,b2b,year,month,day,week,unit_price
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,MUMBAI,MAHARASHTRA,400081,IN,False,2022,4,30,17,0.0
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,BENGALURU,KARNATAKA,560085,IN,False,2022,4,30,17,406.0
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,NAVI MUMBAI,MAHARASHTRA,410210,IN,True,2022,4,30,17,329.0
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,PUDUCHERRY,PUDUCHERRY,605008,IN,False,2022,4,30,17,0.0
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,CHENNAI,TAMIL NADU,600073,IN,False,2022,4,30,17,574.0


In [92]:
df_copy['order_type'] = df['b2b'].map({True:"B2B", False: "B2C"})

In [93]:
df_copy.head()

,order id,date,status,fulfilment,sales channel,ship-service-level,style,sku,category,size,...,ship-state,ship-postal-code,ship-country,b2b,year,month,day,week,unit_price,order_type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,MAHARASHTRA,400081,IN,False,2022,4,30,17,0.0,B2C
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,KARNATAKA,560085,IN,False,2022,4,30,17,406.0,B2C
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,MAHARASHTRA,410210,IN,True,2022,4,30,17,329.0,B2B
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,PUDUCHERRY,605008,IN,False,2022,4,30,17,0.0,B2C
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,TAMIL NADU,600073,IN,False,2022,4,30,17,574.0,B2C


In [94]:
df_copy['sales channel '].value_counts()

,count
sales channel,
Amazon.in,128845
Non-Amazon,124


During the inspection, I noticed that some column names contain unnecessary leading or trailing whitespace.  
To ensure consistent column referencing and avoid potential errors during analysis, I remove these spaces using `str.strip()`.

In [95]:
df_copy.columns = df_copy.columns.str.strip()

In [96]:
df_copy.columns

Index(['order id', 'date', 'status', 'fulfilment', 'sales channel',
       'ship-service-level', 'style', 'sku', 'category', 'size', 'asin',
       'courier status', 'qty', 'currency', 'amount', 'ship-city',
       'ship-state', 'ship-postal-code', 'ship-country', 'b2b', 'year',
       'month', 'day', 'week', 'unit_price', 'order_type'],
      dtype='object')

In [97]:
df_copy.head()

,order id,date,status,fulfilment,sales channel,ship-service-level,style,sku,category,size,...,ship-state,ship-postal-code,ship-country,b2b,year,month,day,week,unit_price,order_type
Unnamed: 0,,,,,,,,,,,,,,,,,,,,,
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,...,MAHARASHTRA,400081,IN,False,2022,4,30,17,0.0,B2C
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,...,KARNATAKA,560085,IN,False,2022,4,30,17,406.0,B2C
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,...,MAHARASHTRA,410210,IN,True,2022,4,30,17,329.0,B2B
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,...,PUDUCHERRY,605008,IN,False,2022,4,30,17,0.0,B2C
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,...,TAMIL NADU,600073,IN,False,2022,4,30,17,574.0,B2C
